# 02 · launch the sweeps
`grids.first_pass()` = gradient statistics at 4 particle counts × 2 shapes, the baseline, and one run per knob value
(seeds, particles, Fourier modes, GRAD_INIT_FRAC). Jobs run as **detached processes** through a scheduler that also survives
closing this notebook; every job is skipped if its `DONE` marker exists, so re-launching is always safe.

In [ ]:
import os, sys, json, time
from pathlib import Path
HERE = Path.cwd()                       # this notebook lives in experiments/simple
assert (HERE / "runInflow.py").exists(), "run this notebook from experiments/simple (Jupyter's cwd is the notebook's folder)"
sys.path.insert(0, str(HERE))
ROOT = HERE / "sweep_results"           # results root (committed to git; warm_cache/ and smoke/ are ignored)
import runInflow as ri
from sweep import config as C, grids, launcher, analysis
print("cwd:", HERE, "| results:", ROOT)

In [ ]:
configs = grids.first_pass()
import pandas as pd
rows = []
for cfg in configs:
    r = C.resolve(cfg, ri); seeds = C.seeds_of(r) if r["kind"] == "optimise" else r["gradstat"]["seeds"]
    nw = C.n_workers_of(r, seeds)
    rows.append(dict(kind=r["kind"], group=r["group"], name=r["name"], N_REF=r["params"]["N_REF"], seeds=len(seeds),
                     N_FOURIER=r["params"]["N_FOURIER"], frac=r["params"]["GRAD_INIT_FRAC"], workers=nw, mem_gb=round(C.mem_estimate_gb(r, nw), 1)))
pd.DataFrame(rows)

In [ ]:
# write the queue (run dirs + config.json), then start the detached scheduler.
# max_cores: physical cores you want to occupy (fibonacci has 16); mem_budget_gb: leave headroom for other users.
qf = launcher.make_queue(configs, ROOT, ri)
pid = launcher.start_detached(qf, max_cores=16, mem_budget_gb=60)
print("queue:", qf, "\nscheduler pid:", pid, "\nlog:", qf.with_suffix(".launcher.log"))

In [ ]:
# status (re-run this cell any time; the scheduler keeps going without the notebook)
pd.set_option("display.width", 200)
st = launcher.status(ROOT)
st[st.group != "smoke"].drop(columns=["run_dir"])

In [ ]:
# log of one run
import subprocess
print(subprocess.run(["tail", "-5", str(qf.with_suffix(".launcher.log"))], capture_output=True, text=True).stdout)
st = launcher.status(ROOT); running = st[st.state == "running"]
for rd in running.run_dir.head(3):
    print("====", rd); print(launcher.tail(rd, 8))

### Phase 2 (later): replicates with other seed sets
Across-run mean/variance of the optimised coefficients needs replicates. When the first pass is done:
```python
qf2 = launcher.make_queue(grids.phase2_replicates(), ROOT, ri)
launcher.start_detached(qf2, max_cores=16, mem_budget_gb=60)
```
### Stopping things
```python
!pkill -f "sweep.launcher"      # stop the scheduler (running jobs continue)
!pkill -f "sweep.run"           # stop the jobs too (progress.json shows 'killed'; re-launch reruns them)
```